# Lecture 6: Statistical Modelling Part I

:::{admonition} Learning Objectives
:class: tip
After this lecture, you will be able to:
- Describe the pillars of effective modelling (data quality, constraints, objectives)
- Choose between parametric and non-parametric approaches
- Distinguish regression, classification, supervised, and unsupervised tasks
- Use scikit-learn's estimator API consistently
- Implement cross-validation for honest model evaluation
- Apply modelling concepts to a real-world case study
:::

## Table of Contents

- [The Pillars of Effective Modelling](#the-pillars-of-effective-modelling)
- [Choice of Modelling Approach](#choice-of-modelling-approach)
- [Scikit-learn's Estimator API](#scikit-learns-estimator-api)
- [Cross-Validation](#cross-validation)
- [Case Study: Regression with Structured Losses](#case-study-regression-with-structured-losses)
- [SWE: Testing & Debugging](#swe-testing--debugging)
- [Exercises](#exercises)
- [Further Reading](#further-reading)

This lecture builds on the feature-engineering pipeline established in [Lecture 5](lecture_5.ipynb) and prepares the ground for the model selection and regularisation material in [Lecture 7](lecture_7.ipynb). The framing of prediction vs. inference echoes the "two cultures" discussion introduced in [Lecture 1](lecture_1.md) {cite}`breiman2001statistical`. Standard references throughout: {cite}`hastie2009elements,james2021introduction`; scikit-learn {cite}`pedregosa2011scikit` is our reference implementation.

## The Pillars of Effective Modelling

The model can only be as good as its ingredients. Whether we frame our work as *data modelling* (inferring the mechanism) or *algorithmic modelling* (optimising predictive accuracy) {cite}`breiman2001statistical`, three prerequisites remain: understand the data, understand the constraints, and specify an objective consistent with the decision the model informs.

:::{note}
If we build a model based on a wrong understanding of the business objective or on low-quality data, any time spent on feature engineering or tuning is likely worthless.
:::

### Know your data

Before fitting any estimator, you should have:
- Examined the target variable (distribution, outliers)
- Checked for systematically missing observations
- A good understanding of the data-generating process
- Thought about potential data drift

### Know your constraints

- Which features are available at prediction time?
- Are there sensitive or discriminatory features we cannot use?
- Monotonicity constraints (e.g., price must increase with quality)?
- Interaction constraints (e.g., linearity in prices)?

### Choose your objective function

We project our target $Y$ onto some feature space via $f(X)$. Key questions:
- What distribution fits the target? (normal, log-normal, Poisson, gamma)
- Do we care about the mean (MSE) or quantiles (MAE, asymmetric loss)?
- Do we want robustness against outliers (Huber loss)?
- Are we optimising a policy function informed by the estimator?

## Choice of Modelling Approach
Key distinctions to guide your choice:

| Dimension | Options |
|-----------|----------|
| Parametric vs Non-Parametric | Linear models vs trees/kernels |
| Supervised vs Unsupervised | Labelled targets vs structure discovery |
| Regression vs Classification | Continuous vs categorical target |
| Prediction vs Causal Identification | Forecasting vs understanding mechanisms |

There is always a trade-off between prediction accuracy and interpretability.

### Going beyond the simple linear model

We can order the workhorse regression families by the objective they minimise. Let $y \in \mathbb{R}^n$ be the target and $X \in \mathbb{R}^{n \times p}$ the design matrix.

- **Ordinary Least Squares (OLS)** — unregularised linear regression:

$$\min_{\beta} \; \|y - X\beta\|_2^2.$$

- **Ridge** — $\ell_2$-penalised OLS, shrinks coefficients and stabilises collinear designs:

$$\min_{\beta} \; \|y - X\beta\|_2^2 + \lambda \|\beta\|_2^2.$$

- **Lasso** — $\ell_1$ penalty, encourages sparsity:

$$\min_{\beta} \; \|y - X\beta\|_2^2 + \lambda \|\beta\|_1.$$

- **Generalised Linear Models (GLMs)** {cite}`nelder1972glm,mccullagh1989generalized` extend OLS to non-Gaussian responses via a link function $g$ and an exponential-family likelihood. For the Poisson GLM with log link, the negative log-likelihood (up to constants) is

$$\ell(\beta) = \sum_{i=1}^n \left[ \exp(x_i^\top \beta) - y_i \, x_i^\top \beta \right].$$

  Model quality is assessed via the **deviance** $D = 2\{\ell(y; y) - \ell(\hat\mu; y)\}$, the GLM analogue of residual sum of squares.

- **Trees** partition the feature space; **random forests** {cite}`breiman2001random` average bagged trees to reduce variance; **gradient boosting** fits an additive expansion of shallow trees to the negative-gradient of the loss.

The progression trades interpretability for flexibility. Below we compare all of them under a common evaluation protocol.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_validate

from fun_ds.data import load_california_housing
from fun_ds.plotting import set_lecture_style

set_lecture_style()

df = load_california_housing()
X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]

models = {
    "Linear":            LinearRegression(),
    "Ridge":             Ridge(alpha=1.0),
    "Decision Tree":     DecisionTreeRegressor(max_depth=10, random_state=0),
    "Random Forest":     RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=0),
}

rows = []
for name, est in models.items():
    cv = cross_validate(
        est, X, y, cv=5,
        scoring=["r2", "neg_root_mean_squared_error"],
        n_jobs=-1,
    )
    rows.append({
        "model": name,
        "R2 (mean)":   cv["test_r2"].mean(),
        "R2 (std)":    cv["test_r2"].std(),
        "RMSE (mean)": -cv["test_neg_root_mean_squared_error"].mean(),
        "RMSE (std)":  cv["test_neg_root_mean_squared_error"].std(),
    })

results = (
    pd.DataFrame(rows)
    .set_index("model")
    .sort_values("R2 (mean)", ascending=False)
)
results.round(3)

## Scikit-learn's Estimator API
All scikit-learn estimators follow a consistent interface:
- `.fit(X, y)` — learn from data
- `.predict(X)` — make predictions
- `.score(X, y)` — evaluate performance
- `.get_params()` / `.set_params()` — hyperparameter access

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

model = Ridge(alpha=1.0)
model.fit(X_train, y_train)               # learn parameters

y_pred = model.predict(X_test[:5])        # apply to new data
print("First 5 predictions:", np.round(y_pred, 3))

print(f"Train R2: {model.score(X_train, y_train):.4f}")
print(f"Test  R2: {model.score(X_test,  y_test):.4f}")

# Hyperparameter introspection — same contract for every sklearn estimator.
print("Params:", model.get_params())

## Cross-Validation

A single train/test split gives a noisy estimate of model performance. Cross-validation {cite}`stone1974cross` provides a more robust assessment by averaging held-out scores across multiple splits.

**K-Fold CV:**
1. Split data into $K$ folds.
2. For each fold: train on $K-1$ folds, evaluate on the held-out fold.
3. Report mean and standard deviation of the metric.

:::{admonition} Formal statement
:class: important
Let $\hat f^{(-k)}$ denote the model trained with the $k$-th fold held out, and let $L(\cdot, \cdot)$ be a bounded loss. The $K$-fold CV estimator of the expected prediction error is

$$\widehat{\mathrm{Err}}_{\mathrm{CV}} = \frac{1}{n} \sum_{k=1}^K \sum_{i \in \mathcal{F}_k} L\!\left(y_i, \hat f^{(-k)}(x_i)\right).$$

Under mild regularity conditions, $\widehat{\mathrm{Err}}_{\mathrm{CV}}$ is an **approximately unbiased estimator of the expected test error** $\mathrm{Err} = E[L(Y, \hat f(X))]$ evaluated at a training-set size of $n(1 - 1/K)$ — a small pessimistic bias that vanishes as $K \to n$ (leave-one-out CV) {cite}`stone1974cross`. Its variance decomposes as

$$\mathrm{Var}(\widehat{\mathrm{Err}}_{\mathrm{CV}}) = \frac{1}{n}\sigma^2 + \frac{n-1}{n}\bar\rho\,\sigma^2,$$

where $\bar\rho$ is the average correlation between fold-level errors: large $K$ reduces the first term but drives up the correlation $\bar\rho$, which is why leave-one-out CV has notoriously high variance despite being nearly unbiased. {cite:t}`kohavi1995study` empirically established $K = 5$ or $K = 10$ as the practical sweet spot; this is still the default in scikit-learn.
:::

:::{admonition} Forward signal — D200/D300
:class: seealso
We use cross-validation here as a practical tool for honest performance estimates. **D300** formalises why it works: the CV score is an approximately unbiased estimator of the expected test error, and its variance is what the bias-variance decomposition explains.
:::

:::{admonition} Bias-variance decomposition (squared-error loss)
:class: note
For a regression problem with additive noise $Y = f(X) + \varepsilon$, $E[\varepsilon] = 0$, $\mathrm{Var}(\varepsilon) = \sigma^2$, the expected squared prediction error at a fixed point $x$ decomposes as {cite}`hastie2009elements`:

$$\mathbb{E}\!\left[(Y - \hat f(x))^2 \mid X = x\right] = \underbrace{\sigma^2}_{\text{irreducible}} + \underbrace{\bigl(\mathbb{E}[\hat f(x)] - f(x)\bigr)^2}_{\text{Bias}^2(\hat f)} + \underbrace{\mathbb{E}\!\left[\bigl(\hat f(x) - \mathbb{E}[\hat f(x)]\bigr)^2\right]}_{\text{Var}(\hat f)}.$$

The irreducible term $\sigma^2$ is a property of the data-generating process; no learner can beat it. The bias and variance terms are properties of the *estimator*: increasing model complexity (more features, deeper trees, less regularisation) typically lowers bias but raises variance. The whole point of model selection — cross-validation, regularisation strength tuning, etc. — is to navigate this trade-off. See {cite:t}`hastie2009elements` §7.3.
:::

In [ ]:
from fun_ds.evaluation import cross_val_summary

summary = cross_val_summary(
    Ridge(alpha=1.0), X, y, cv=5,
    scoring=["r2", "neg_mean_squared_error"],
)
summary.round(4)

### Stratified and grouped CV

:::{important}
Standard K-Fold may not be appropriate when:
- Classes are imbalanced (use StratifiedKFold)
- Data has group structure, e.g., multiple observations per entity (use GroupKFold)
- Data is temporal (use TimeSeriesSplit)
:::

## Case Study: Regression with Structured Losses

Real-world regression rarely calls for plain squared-error loss. Insurance pricing is the archetype:

- Claim **frequency** is a non-negative count $\Rightarrow$ Poisson GLM with log link.
- Claim **severity** is a positive skewed amount $\Rightarrow$ Gamma GLM with log link.
- Business logic imposes **monotonicity** constraints (e.g. price non-decreasing in risk).

:::{admonition} Why GLMs are the actuarial industry standard
:class: note
Even in an era when GBDTs win most tabular benchmarks, **Generalised Linear Models** {cite}`nelder1972glm,mccullagh1989generalized` remain the dominant tool for insurance pricing. The reasons are institutional as much as statistical:

1. **Interpretability of coefficients.** Under a log link, $\exp(\hat\beta_j)$ is the multiplicative effect of a unit change in feature $j$ on the predicted mean — an actuary can point to a single number and say "young drivers pay 40% more because of this coefficient."
2. **Regulatory justification.** Under EU Solvency II, the US NAIC rate-filing regime, and analogous frameworks worldwide, insurers must demonstrate to a regulator that premium differentials are actuarially sound and not unlawfully discriminatory. A GLM with a small, auditable set of features is far easier to justify than a black-box GBDT with thousands of splits.
3. **Additivity in the log-mean.** GLMs decompose the predicted premium as a product of factors ("base rate × age factor × region factor × vehicle factor"), which matches how underwriters and brokers already think about pricing.
4. **Well-understood inference.** Standard errors, deviance-based hypothesis tests, and pricing-adequacy diagnostics have decades of accepted practice behind them.

GBDTs are increasingly used *alongside* GLMs — for pricing sophistication studies, residual analysis, or as a challenger model — but the filed rate itself is nearly always a GLM.
:::

The canonical benchmark is the French Motor Third-Party Liability dataset, but it requires an OpenML fetch that may not be available in every environment. We attempt it below, and otherwise fall back to a structured-loss demo on California Housing: predict `MedHouseVal` treated as a positive continuous target using Poisson and Gamma regression, and enforce monotonicity with `HistGradientBoostingRegressor`.

In [ ]:
from sklearn.linear_model import PoissonRegressor, GammaRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_poisson_deviance, mean_gamma_deviance

# Try to load French Motor TPL from OpenML; fall back to California housing.
try:
    from sklearn.datasets import fetch_openml
    tpl = fetch_openml(data_id=41214, as_frame=True, parser="pandas")
    X_case = tpl.data.select_dtypes("number").fillna(0.0)
    y_case = tpl.data["ClaimNb"] / tpl.data["Exposure"].clip(lower=1e-3)
    print(f"Loaded French Motor TPL: {X_case.shape}")
except Exception as exc:  # network, OpenML outage, etc.
    print(f"OpenML fetch failed ({type(exc).__name__}); falling back to California Housing.")
    X_case = X.copy()
    y_case = y.copy()  # MedHouseVal is strictly positive

# --- Poisson GLM on log-linked mean --------------------------------------
# Poisson admits y >= 0, so it is safe for both datasets.
poisson_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("glm",    PoissonRegressor(alpha=1e-3, max_iter=500)),
])
poisson_pipe.fit(X_case, y_case)
y_hat_poisson = np.clip(poisson_pipe.predict(X_case), 1e-6, None)
print(f"Poisson deviance: {mean_poisson_deviance(y_case, y_hat_poisson):.4f}")

# --- Gamma GLM (strictly positive targets only) --------------------------
# Gamma requires y > 0. Restrict to the positive subset (equivalent to a
# hurdle model's severity component when running on insurance data).
pos_mask = np.asarray(y_case) > 0
X_pos, y_pos = X_case.loc[pos_mask], y_case[pos_mask]
gamma_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("glm",    GammaRegressor(alpha=1e-3, max_iter=500)),
])
gamma_pipe.fit(X_pos, y_pos)
y_hat_gamma = np.clip(gamma_pipe.predict(X_pos), 1e-6, None)
print(f"Gamma deviance (positive-only, n={pos_mask.sum()}): "
      f"{mean_gamma_deviance(y_pos, y_hat_gamma):.4f}")

# --- Monotonicity-constrained gradient boosting --------------------------
# Enforce non-decreasing prediction in the first informative feature.
monotone = np.zeros(X_case.shape[1], dtype=int)
mono_feature = "MedInc" if "MedInc" in X_case.columns else X_case.columns[0]
monotone[X_case.columns.get_loc(mono_feature)] = 1

hgb = HistGradientBoostingRegressor(
    loss="poisson",
    max_iter=200,
    monotonic_cst=monotone,
    random_state=0,
)
scores = cross_val_score(hgb, X_case, y_case, cv=5,
                         scoring="neg_mean_poisson_deviance", n_jobs=-1)
print(f"HGB (Poisson loss, monotone in {mono_feature!r}) "
      f"mean CV deviance: {-scores.mean():.4f}")

### Model diagnostics

Before shipping a fitted model, we routinely inspect a handful of diagnostics that are well established in the classical regression toolbox {cite}`hastie2009elements`.

- **Residual plots.** Plot residuals $r_i = y_i - \hat y_i$ against fitted values $\hat y_i$ and against each feature. A well-specified linear model produces a roughly horizontal band around zero; systematic curvature indicates missing non-linearities, a funnel shape indicates heteroscedasticity (consider a variance-stabilising link or a Gamma/Poisson GLM), and clusters of same-signed residuals indicate an omitted grouping variable.
- **Cook's distance.** For observation $i$, Cook's $D_i$ measures how much the vector of fitted values would change if $i$ were removed:
  $$D_i = \frac{r_i^2}{p \, \hat\sigma^2} \cdot \frac{h_{ii}}{(1 - h_{ii})^2},$$
  where $h_{ii}$ is the leverage (diagonal of the hat matrix $H = X(X^\top X)^{-1}X^\top$) and $p$ is the number of parameters. A rough rule of thumb flags $D_i > 4/n$ as **influential**; such points deserve a manual look before you keep or drop them.
- **Variance Inflation Factor (VIF).** For feature $j$, fit an auxiliary regression of $x_j$ on all other features and let $R_j^2$ be its coefficient of determination; then
  $$\mathrm{VIF}_j = \frac{1}{1 - R_j^2}.$$
  A common heuristic treats $\mathrm{VIF}_j > 5$ (or $10$) as evidence of problematic multicollinearity: the corresponding coefficient's standard error is inflated by the same factor, so its estimated sign can flip under trivial perturbations. Remedies include dropping the redundant feature, combining collinear features into a summary (e.g. via PCA), or switching to a regularised estimator (Ridge, Elastic Net).

For GLMs and tree ensembles, the analogous diagnostics are the **deviance residuals** and **partial-dependence / permutation-importance** plots — we return to the latter in [Lecture 8](lecture_8.ipynb) on explainability.

:::{note}
**The No Free Lunch theorem.** {cite:t}`wolpert1996nofreelunch` proved that, averaged uniformly over *all* possible target functions, every learning algorithm has the same expected off-training-set error — including a coin flip. In other words, **no algorithm dominates all others across every conceivable problem**. Any observed superiority of, say, gradient boosting over linear regression on tabular data is a statement about the *distribution of real-world problems*, not about the algorithms in the abstract.

The practical implication for this course is that model comparison must always be empirical and problem-specific: benchmark several families (linear, tree ensemble, GBDT, and — from [Lecture 7](lecture_7.ipynb) — tabular foundation models) on your own data and pick the one that generalises best under an honest cross-validation protocol.
:::

## SWE: Testing & Debugging
Key practices:
- Write unit tests for data transformations and model pipelines
- Use `pytest` fixtures to share test data
- Debug with breakpoints (`breakpoint()`) rather than print statements
- Test edge cases: empty DataFrames, NaN values, unseen categories

## Exercises

Work through the following on your own. Solutions are discussed in the tutorial session; a reference implementation lives in `notebooks/solutions/lecture_6/`.

:::{admonition} Exercise 1 — Sensitivity of CV to the number of folds
:class: tip
Using `load_california_housing()` and a `Ridge(alpha=1.0)` estimator, compute the mean and standard deviation of the R² score under $K \in \{2, 5, 10, 20\}$ folds via `cross_val_summary`. Plot the mean $\pm$ std as $K$ increases. What do you observe about bias vs. variance of the estimator as $K$ grows? Relate your answer to the discussion signposted for D300.
:::

:::{admonition} Exercise 2 — StratifiedKFold on a discretised target
:class: tip
The California housing target is continuous, so vanilla K-Fold gives folds with different mean prices. Bin `MedHouseVal` into quintiles (`pd.qcut(y, 5)`) and use `StratifiedKFold(n_splits=5)` over these bins to evaluate `RandomForestRegressor(n_estimators=100)`. Compare the fold-level R² spread against plain K-Fold. When is stratification worth the extra bookkeeping?
:::

:::{admonition} Exercise 3 — A Poisson regression pipeline
:class: tip
Build a `Pipeline` that (i) standardises numerical features, (ii) fits `PoissonRegressor` on California housing with `MedHouseVal` as the target. Report the cross-validated Poisson deviance. Then print the fitted coefficients as *multiplicative* effects ($\exp(\hat\beta_j)$ per one-standard-deviation increase in the feature) and interpret the top three drivers. Would you draw a causal conclusion from these coefficients? Why or why not — cite {cite}`breiman2001statistical`.
:::

## Further Reading

Statistical modelling has an unusually rich set of authoritative textbook treatments; pick the one that matches your background and skim the relevant chapters when a topic in this lecture (or in [Lecture 7](lecture_7.ipynb)) is unclear.

- {cite:t}`hastie2009elements` — *The Elements of Statistical Learning*. The reference textbook for supervised learning; especially strong on GLMs (Ch. 4), model assessment (Ch. 7), and additive models (Ch. 9).
- {cite:t}`james2021introduction` — *An Introduction to Statistical Learning*. A gentler-paced companion to Hastie et al., with worked R (and, in the newer edition, Python) code.
- {cite:t}`bishop2006pattern` — *Pattern Recognition and Machine Learning*. A Bayesian-flavoured tour of the same material; the definitive treatment of probabilistic classifiers and mixture models.
- {cite:t}`murphy2022probabilistic` — *Probabilistic Machine Learning: An Introduction*. A modern, unified probabilistic viewpoint spanning classical ML and deep learning; freely available online.
- {cite:t}`goodfellow2016deep` — *Deep Learning*. Useful now as a reference for the optimisation intuition (SGD, momentum, second-order methods) that transfers to GBMs; the primary reference for the neural-network content D200 will cover.

For MPhil EDS students specifically, {cite:t}`varian2014bigdata`, {cite:t}`mullainathan2017machine`, and {cite:t}`athey2019machine` bridge these methods back to econometric practice (see also [Lecture 1 §2](lecture_1.md#prediction-inference-and-causation)).

:::{admonition} Key Takeaways
:class: important
- Data quality and correct problem formulation matter more than model choice
- Know your constraints before modelling (available features, fairness, monotonicity)
- Choose objective functions that align with the business goal
- Use cross-validation for honest performance estimates
- The scikit-learn API provides a consistent interface across all model families
:::